In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:

import os, time, warnings
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Ellipse
from matplotlib.colors import Normalize
import pandas as pd

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

# ─────────────────────────────────────────────────────────────────
# 0.  Global configuration
# ─────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


Using device: cpu


In [2]:
# Domain
X_MIN, X_MAX = -6.0, 15.0
Y_MIN, Y_MAX =  -6.0,  6.0
A_ELLIPSE, B_ELLIPSE = 1.0, 0.5   # semi-axes

# True Reynolds number used to generate synthetic data
RE_TRUE = 100.0

# Training hyper-parameters
HIDDEN_LAYERS = 6
HIDDEN_NODES  = 64
EPOCHS_FORWARD  = 3000   # forward PINN to generate synthetic data
EPOCHS_INVERSE  = 4000  # inverse PINN per N sweep
LR              = 1e-3
LAMBDA_PDE      = 1.0
LAMBDA_BC       = 10.0
LAMBDA_DATA     = 10.0
CONVERGENCE_TOL = 0.10     # 10 % error threshold

OUTPUT_DIR = "problem3_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# ─────────────────────────────────────────────────────────────────
# 1.  Helper utilities
# ─────────────────────────────────────────────────────────────────

def inside_ellipse(x, y, a=A_ELLIPSE, b=B_ELLIPSE):
    """Return boolean mask: True if (x,y) is inside the ellipse."""
    return (x / a) ** 2 + (y / b) ** 2 < 1.0


def sample_domain(n, rng=None):
    """Sample n points uniformly in the domain, excluding the ellipse interior."""
    if rng is None:
        rng = np.random.default_rng(42)
    pts = []
    while len(pts) < n:
        x = rng.uniform(X_MIN, X_MAX, n * 3)
        y = rng.uniform(Y_MIN, Y_MAX, n * 3)
        mask = ~inside_ellipse(x, y)
        pts.extend(zip(x[mask], y[mask]))
    pts = np.array(pts[:n], dtype=np.float32)
    return pts

def to_tensor(arr, grad=False):
    t = torch.tensor(arr, dtype=torch.float32, device=DEVICE, requires_grad=grad)
    return t


def draw_ellipse_patch(ax, **kwargs):
    e = Ellipse((0, 0), 2 * A_ELLIPSE, 2 * B_ELLIPSE,
                color="white", zorder=5, **kwargs)
    ax.add_patch(e)
    e2 = Ellipse((0, 0), 2 * A_ELLIPSE, 2 * B_ELLIPSE,
                 fill=False, edgecolor="k", lw=0.8, zorder=6)
    ax.add_patch(e2)



In [4]:
# ─────────────────────────────────────────────────────────────────
# 2.  Neural-network architecture (shared by forward & inverse)
# ─────────────────────────────────────────────────────────────────

class PINN(nn.Module):
    """
    Fully-connected network: (x, y) -> (u, v, p).
    For the inverse problem, 1/Re is an additional nn.Parameter.
    """

    def __init__(self, layers, nodes, inverse=False, re_init=50.0):
        super().__init__()
        self.inverse = inverse

        # Build MLP with tanh activations
        net = []
        net.append(nn.Linear(2, nodes))
        for _ in range(layers - 1):
            net.append(nn.Tanh())
            net.append(nn.Linear(nodes, nodes))
        net.append(nn.Tanh())
        net.append(nn.Linear(nodes, 3))   # outputs: u, v, p
        self.net = nn.Sequential(*net)

        # Xavier initialisation
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
        if inverse:
            # Parameterise as log(1/Re) to enforce positivity
            init_log = np.log(1.0 / re_init)
            self._log_nu = nn.Parameter(
                torch.tensor(init_log, dtype=torch.float32))

    def forward(self, xy):
        return self.net(xy)

    @property
    def nu(self):
        """Returns 1/Re (kinematic viscosity for unit U_inf, unit length)."""
        return torch.exp(self._log_nu)

    @property
    def Re(self):
        return 1.0 / self.nu



In [5]:
# ─────────────────────────────────────────────────────────────────
# 3.  PDE residuals (steady incompressible 2-D Navier–Stokes)
# ─────────────────────────────────────────────────────────────────

def ns_residuals(model, xy, nu_override=None):
    """
    Compute continuity and x,y-momentum residuals via automatic differentiation.
    nu_override: scalar tensor for forward PINN (fixed Re); None -> use model.nu
    """
    xy = xy.clone().requires_grad_(True)
    out = model(xy)
    u, v, p = out[:, 0:1], out[:, 1:2], out[:, 2:3]

    nu = nu_override if nu_override is not None else model.nu

    def grad(f, x):
        return torch.autograd.grad(f, x,
                                   grad_outputs=torch.ones_like(f),
                                   create_graph=True,
                                   retain_graph=True)[0]

    grads_u = grad(u, xy)
    u_x, u_y = grads_u[:, 0:1], grads_u[:, 1:2]

    grads_v = grad(v, xy)
    v_x, v_y = grads_v[:, 0:1], grads_v[:, 1:2]

    grads_p = grad(p, xy)
    p_x, p_y = grads_p[:, 0:1], grads_p[:, 1:2]
    u_xx = grad(u_x, xy)[:, 0:1]
    u_yy = grad(u_y, xy)[:, 1:2]
    v_xx = grad(v_x, xy)[:, 0:1]
    v_yy = grad(v_y, xy)[:, 1:2]

    cont  = u_x + v_y
    mom_x = u * u_x + v * u_y + p_x - nu * (u_xx + u_yy)
    mom_y = u * v_x + v * v_y + p_y - nu * (v_xx + v_yy)

    return cont, mom_x, mom_y


In [6]:
# ─────────────────────────────────────────────────────────────────
# 4.  Boundary-condition losses
# ─────────────────────────────────────────────────────────────────

def bc_loss(model, nu_fixed=None):
    losses = []

    # --- Inlet: x = X_MIN,  u=1, v=0 ---
    y_in  = torch.linspace(Y_MIN, Y_MAX, 60, device=DEVICE).unsqueeze(1)
    x_in  = torch.full_like(y_in, X_MIN)
    xy_in = torch.cat([x_in, y_in], dim=1)
    out   = model(xy_in)
    losses.append(torch.mean((out[:, 0] - 1.0) ** 2))   # u=1
    losses.append(torch.mean( out[:, 1] ** 2))            # v=0

    # --- Outlet: x = X_MAX,  p=0 ---
    y_out  = torch.linspace(Y_MIN, Y_MAX, 60, device=DEVICE).unsqueeze(1)
    x_out  = torch.full_like(y_out, X_MAX)
    xy_out = torch.cat([x_out, y_out], dim=1)
    out_o  = model(xy_out)
    losses.append(torch.mean(out_o[:, 2] ** 2))           # p=0

    # --- Top/bottom: slip (v=0) ---
    x_tb   = torch.linspace(X_MIN, X_MAX, 80, device=DEVICE).unsqueeze(1)
    y_top  = torch.full_like(x_tb, Y_MAX)
    y_bot  = torch.full_like(x_tb, Y_MIN)
    for yy in [y_top, y_bot]:
        xy_tb = torch.cat([x_tb, yy], dim=1)
        out_tb = model(xy_tb)
        losses.append(torch.mean(out_tb[:, 1] ** 2))      # v=0

    # --- Ellipse surface: no-slip (u=v=0) ---
    theta = torch.linspace(0, 2 * np.pi, 120, device=DEVICE)
    x_e   = A_ELLIPSE * torch.cos(theta).unsqueeze(1)
    y_e   = B_ELLIPSE * torch.sin(theta).unsqueeze(1)
    xy_e  = torch.cat([x_e, y_e], dim=1)
    out_e = model(xy_e)
    losses.append(torch.mean(out_e[:, 0] ** 2))           # u=0
    losses.append(torch.mean(out_e[:, 1] ** 2))           # v=0

    return sum(losses)



In [9]:
# ─────────────────────────────────────────────────────────────────
# 5.  Forward PINN (to generate synthetic "measurement" data)
# ─────────────────────────────────────────────────────────────────

def train_forward_pinn(re_true, n_col=3000, epochs=EPOCHS_FORWARD, verbose=True):
    """Train a forward PINN at a fixed Re and return the trained model."""
    nu_fixed = torch.tensor(1.0 / re_true, dtype=torch.float32, device=DEVICE)

    model = PINN(HIDDEN_LAYERS, HIDDEN_NODES, inverse=False).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=5000, gamma=0.5)

    # Collocation points (interior)
    col_np  = sample_domain(n_col)
    xy_col  = to_tensor(col_np, grad=False)

    history = []
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()

        cont, mx, my = ns_residuals(model, xy_col, nu_override=nu_fixed)
        loss_pde = torch.mean(cont**2 + mx**2 + my**2)
        loss_bc  = bc_loss(model, nu_fixed=nu_fixed)
        loss     = LAMBDA_PDE * loss_pde + LAMBDA_BC * loss_bc

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()

        history.append(loss.item())
        if verbose and ep % 2000 == 0:
            print(f"  [Forward Re={re_true:.0f}] ep {ep:>6d}  "
                  f"loss={loss.item():.4e}  "
                  f"pde={loss_pde.item():.4e}  "
                  f"bc={loss_bc.item():.4e}  "
                  f"t={time.time()-t0:.1f}s")

    return model, history

        

In [10]:
# ─────────────────────────────────────────────────────────────────
# 6.  Inverse PINN training
# ─────────────────────────────────────────────────────────────────

def train_inverse_pinn(obs_xy, obs_uvp,
                       n_col=3000,
                       epochs=EPOCHS_INVERSE,
                       re_init=50.0,
                       verbose=True,
                       label=""):
    """
    Train inverse PINN given observation data.
    Returns (model, recovered_Re, history_dict).
    """
    model = PINN(HIDDEN_LAYERS, HIDDEN_NODES, inverse=True,
                 re_init=re_init).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=6000, gamma=0.5)

    # Collocation points
    col_np = sample_domain(n_col)
    xy_col = to_tensor(col_np, grad=False)

    # Observation tensors
    xy_obs  = to_tensor(obs_xy)
    uvp_obs = to_tensor(obs_uvp)

    hist_total, hist_pde, hist_bc, hist_data, hist_re = [], [], [], [], []
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()

        # Data loss
        pred_obs = model(xy_obs)
        loss_data = torch.mean((pred_obs - uvp_obs) ** 2)

        # PDE loss
        cont, mx, my = ns_residuals(model, xy_col)
        loss_pde = torch.mean(cont**2 + mx**2 + my**2)

        # BC loss
        loss_bc = bc_loss(model)

        loss = (LAMBDA_DATA * loss_data
                + LAMBDA_PDE * loss_pde
                + LAMBDA_BC  * loss_bc)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()

        hist_total.append(loss.item())
        hist_pde.append(loss_pde.item())
        hist_bc.append(loss_bc.item())
        hist_data.append(loss_data.item())
        hist_re.append(model.Re.item())

        if verbose and ep % 2500 == 0:
            print(f"  [{label}] ep {ep:>6d}  "
                  f"total={loss.item():.4e}  "
                  f"Re_pred={model.Re.item():.2f}  "
                  f"Re_true={RE_TRUE:.0f}  "
                  f"t={time.time()-t0:.1f}s")

    recovered_re = model.Re.item()
    history = dict(total=hist_total, pde=hist_pde,
                   bc=hist_bc, data=hist_data, re=hist_re)
    return model, recovered_re, history


In [11]:
# ─────────────────────────────────────────────────────────────────
# 7.  Generate synthetic ground-truth observations
# ─────────────────────────────────────────────────────────────────

def generate_observations(forward_model, pts_xy):
    """Query the forward PINN to get (u,v,p) at given points."""
    forward_model.eval()
    with torch.no_grad():
        xy_t   = to_tensor(pts_xy)
        uvp_t  = forward_model(xy_t)
    return uvp_t.cpu().numpy()

In [12]:
# ─────────────────────────────────────────────────────────────────
# 8.  Evaluation grid
# ─────────────────────────────────────────────────────────────────

def build_eval_grid(nx=200, ny=80):
    """Return meshgrid arrays and flattened tensors for evaluation."""
    xv = np.linspace(X_MIN, X_MAX, nx)
    yv = np.linspace(Y_MIN, Y_MAX, ny)
    XX, YY = np.meshgrid(xv, yv)
    mask = inside_ellipse(XX, YY)
    xy_flat = np.stack([XX.ravel(), YY.ravel()], axis=1).astype(np.float32)
    return XX, YY, mask, xy_flat

In [13]:
def predict_grid(model, xy_flat):
    """Run model on flat grid and return (u, v, p) arrays."""
    model.eval()
    bs = 4096
    preds = []
    with torch.no_grad():
        for i in range(0, len(xy_flat), bs):
            chunk = to_tensor(xy_flat[i:i+bs])
            preds.append(model(chunk).cpu().numpy())
    return np.vstack(preds)

In [14]:
# ─────────────────────────────────────────────────────────────────
# 9.  Plotting helpers
# ─────────────────────────────────────────────────────────────────

CMAP_U  = "RdBu_r"
CMAP_V  = "seismic"
CMAP_P  = "coolwarm"

def plot_flow_fields(model, title_suffix, fname, re_label=""):
    XX, YY, mask, xy_flat = build_eval_grid()
    uvp = predict_grid(model, xy_flat)
    UU = uvp[:, 0].reshape(XX.shape)
    VV = uvp[:, 1].reshape(XX.shape)
    PP = uvp[:, 2].reshape(XX.shape)
    UU[mask] = np.nan
    VV[mask] = np.nan
    PP[mask] = np.nan

    fig = plt.figure(figsize=(18, 14))
    fig.patch.set_facecolor("#0d0d1a")
    gs  = gridspec.GridSpec(2, 2, figure=fig,
                            hspace=0.38, wspace=0.28)
    titles = ["Streamwise Velocity  $u$",
              "Cross-stream Velocity  $v$",
              "Pressure  $p$",
              "Streamlines"]
    cmaps  = [CMAP_U, CMAP_V, CMAP_P, None]
    fields = [UU, VV, PP, None]


    for idx, (ttl, cmap, fld) in enumerate(zip(titles, cmaps, fields)):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])
        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values():
            sp.set_color("#444")

        if fld is not None:
            vmax = np.nanpercentile(np.abs(fld), 98)
            vmin = -vmax if cmap in [CMAP_V, CMAP_P] else 0
            cf = ax.contourf(XX, YY, fld, levels=50,
                             cmap=cmap, vmin=vmin, vmax=vmax)
            cb = fig.colorbar(cf, ax=ax, fraction=0.025, pad=0.02)
            cb.ax.tick_params(colors="white", labelsize=7)
        else:
            # Streamlines
            speed = np.sqrt(UU**2 + VV**2)
            sp2 = np.where(np.isnan(speed), 0, speed)
            ax.contourf(XX, YY, sp2, levels=40,
                        cmap="magma", alpha=0.7)
            seed_x = np.full(20, X_MIN + 0.05)
            seed_y = np.linspace(Y_MIN + 0.3, Y_MAX - 0.3, 20)
            ax.streamplot(XX[0], YY[:, 0],
                          np.where(np.isnan(UU), 0, UU),
                          np.where(np.isnan(VV), 0, VV),
                          color="white", linewidth=0.5, density=2,
                          start_points=np.c_[seed_x, seed_y],
                          arrowsize=0.5)

        draw_ellipse_patch(ax)
        ax.set_xlim(X_MIN, X_MAX)
        ax.set_ylim(Y_MIN, Y_MAX)
        ax.set_title(ttl, color="white", fontsize=10, pad=6)
        ax.set_xlabel("x", color="#aaa", fontsize=8)
        ax.set_ylabel("y", color="#aaa", fontsize=8)
        ax.tick_params(colors="#aaa", labelsize=7)

    sup = f"Problem 3 — Inverse PINN  |  {title_suffix}"
    if re_label:
        sup += f"  |  {re_label}"
    fig.suptitle(sup, color="white", fontsize=12, y=0.97)
    plt.savefig(os.path.join(OUTPUT_DIR, fname),
                dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {fname}")


def plot_re_convergence(n_list, re_list, err_list, fname):
    fig, ax1 = plt.subplots(figsize=(9, 5))
    fig.patch.set_facecolor("#0d0d1a")
    ax1.set_facecolor("#0d0d1a")

    color_re  = "#f5a623"
    color_err = "#7ed6df"

    ax1.plot(n_list, re_list, "o-", color=color_re,
             lw=2, ms=7, label=f"Recovered Re  (True={RE_TRUE:.0f})")
    ax1.axhline(RE_TRUE, color=color_re, ls="--", lw=1, alpha=0.5)
    ax1.set_xlabel("N  (number of observation points)", color="#ccc", fontsize=11)
    ax1.set_ylabel("Recovered Re", color=color_re, fontsize=11)
    ax1.tick_params(axis="y", colors=color_re)
    ax1.tick_params(axis="x", colors="#ccc")
    ax1.set_title("Inverse PINN — Reynolds Number Recovery vs N",
                  color="white", fontsize=13)
    for sp in ax1.spines.values():
        sp.set_color("#444")

    ax2 = ax1.twinx()
    ax2.set_facecolor("#0d0d1a")
    ax2.plot(n_list, err_list, "s--", color=color_err,
             lw=2, ms=7, label="% Error")
    ax2.axhline(10.0, color=color_err, ls=":", lw=1, alpha=0.6,
                label="10% threshold")
    ax2.set_ylabel("Percentage Error (%)", color=color_err, fontsize=11)
    ax2.tick_params(axis="y", colors=color_err)
    for sp in ax2.spines.values():
        sp.set_color("#444")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2,
               facecolor="#1a1a2e", edgecolor="#444",
               labelcolor="white", fontsize=9,
               loc="upper right")

    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, fname),
                dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {fname}")


In [15]:
def plot_sampling_comparison(n_conv, results_uniform, results_wake, fname):
    """Bar/line plot comparing uniform vs wake-clustered sampling."""
    ns     = [r["N"]     for r in results_uniform]
    err_u  = [r["error"] for r in results_uniform]
    err_w  = [r["error"] for r in results_wake]

    fig, ax = plt.subplots(figsize=(9, 5))
    fig.patch.set_facecolor("#0d0d1a")
    ax.set_facecolor("#0d0d1a")

    w = 0.35
    xs = np.arange(len(ns))
    ax.bar(xs - w/2, err_u, w, label="Uniform sampling",
           color="#f5a623", alpha=0.85)
    ax.bar(xs + w/2, err_w, w, label="Wake-clustered sampling",
           color="#7ed6df", alpha=0.85)
    ax.axhline(10.0, color="white", ls="--", lw=1.2,
               label="10% convergence threshold")
    ax.set_xticks(xs)
    ax.set_xticklabels([str(n) for n in ns], color="#ccc")
    ax.tick_params(axis="y", colors="#ccc")
    ax.set_xlabel("N  (observation points)", color="#ccc", fontsize=11)
    ax.set_ylabel("% Error in recovered Re", color="#ccc", fontsize=11)
    ax.set_title("Uniform vs Wake-Clustered Sampling — Convergence Comparison",
                 color="white", fontsize=12)
    for sp in ax.spines.values():
        sp.set_color("#444")
    ax.legend(facecolor="#1a1a2e", edgecolor="#444",
              labelcolor="white", fontsize=9)
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, fname),
                dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {fname}")


def plot_loss_history(history, re_true, label, fname):
    ep = np.arange(1, len(history["total"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.patch.set_facecolor("#0d0d1a")
    colors = {"total": "#f5a623", "pde": "#7ed6df",
              "bc": "#a29bfe", "data": "#fd79a8"}
    for ax, keys, title in [
        (axes[0],
         ["total", "pde", "bc", "data"],
         "Loss Components"),
        (axes[1],
         ["re"],
         f"Recovered Re  (True={re_true:.0f})")
    ]:
        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values():
            sp.set_color("#444")
        ax.tick_params(colors="#ccc")
        for k in keys:
            lbl = "Re_pred" if k == "re" else k
            col = colors.get(k, "#ffffff")
            ax.semilogy(ep, history[k], lw=1.2,
                        color=col, label=lbl)
        if "re" in keys:
            ax.set_yscale("linear")
            ax.axhline(re_true, color="white",
                       ls="--", lw=1.0, label=f"Re_true={re_true:.0f}")
        ax.set_xlabel("Epoch", color="#ccc")
        ax.set_title(title, color="white", fontsize=10)
        ax.legend(facecolor="#1a1a2e", edgecolor="#444",
                  labelcolor="white", fontsize=8)
    fig.suptitle(f"Training History — {label}",
                 color="white", fontsize=12)
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, fname),
                dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {fname}")


def plot_obs_location(obs_uniform, obs_wake, fname):
    """Show where observation points are placed."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.patch.set_facecolor("#0d0d1a")
    for ax, pts, title, color in [
        (axes[0], obs_uniform, "Uniform Sampling", "#f5a623"),
        (axes[1], obs_wake,    "Wake-Clustered Sampling", "#7ed6df")
    ]:
        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values():
            sp.set_color("#444")
        ax.scatter(pts[:, 0], pts[:, 1], c=color, s=25,
                   zorder=3, alpha=0.85)
        draw_ellipse_patch(ax)
        ax.set_xlim(X_MIN, X_MAX)
        ax.set_ylim(Y_MIN, Y_MAX)
        ax.set_title(title, color="white", fontsize=11)
        ax.set_xlabel("x", color="#aaa")
        ax.set_ylabel("y", color="#aaa")
        ax.tick_params(colors="#aaa")
    fig.suptitle("Observation Point Locations", color="white", fontsize=13)
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, fname),
                dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved: {fname}")



In [ ]:
# ─────────────────────────────────────────────────────────────────
# 10.  Main execution
# ─────────────────────────────────────────────────────────────────

def main():
    print("\n" + "="*65)
    print("  ME4222E — Problem 3: Inverse PINN")
    print(f"  True Reynolds number: Re = {RE_TRUE:.0f}")
    print("="*65)

    # ── Step A: Train forward PINN to generate synthetic data ──────
    print("\n[A] Training forward PINN to generate synthetic observations...")
    fwd_model, fwd_hist = train_forward_pinn(RE_TRUE, epochs=EPOCHS_FORWARD)
    plot_flow_fields(fwd_model,
                     f"Forward PINN  Re={RE_TRUE:.0f}  (Ground Truth)",
                     "fig1_forward_flow_fields.png",
                     re_label=f"Re={RE_TRUE:.0f}")

    # ── Step B: N-sweep with UNIFORM sampling ─────────────────────
    print("\n[B] Inverse PINN — N-sweep with UNIFORM sampling")
    rng_u = np.random.default_rng(7)

    sweep_results = []   # list of dicts
    converged_N_uniform = None

    for N in range(10, 110, 10):
        print(f"\n  N = {N} (uniform)")
        pts_xy  = sample_domain(N, rng=rng_u)
        pts_uvp = generate_observations(fwd_model, pts_xy)

        inv_model, re_rec, hist = train_inverse_pinn(
            pts_xy, pts_uvp,
            re_init=50.0,
            epochs=EPOCHS_INVERSE,
            verbose=True,
            label=f"Uniform N={N}"
        )

        err = abs(re_rec - RE_TRUE) / RE_TRUE * 100
        final_loss = hist["total"][-1]
        sweep_results.append(dict(N=N, strategy="uniform",
                                  Re_rec=re_rec, error=err,
                                  final_loss=final_loss))
        print(f"  >> N={N}  Re_rec={re_rec:.2f}  Error={err:.2f}%  "
              f"Loss={final_loss:.4e}")

        # Plot loss history for each N
        plot_loss_history(hist, RE_TRUE,
                          f"Uniform N={N}",
                          f"fig_loss_uniform_N{N:03d}.png")

        if err <= CONVERGENCE_TOL * 100 and converged_N_uniform is None:
            converged_N_uniform = N
            print(f"  *** CONVERGED at N={N} (uniform) ***")
            # Save flow field at convergence
            plot_flow_fields(inv_model,
                             f"Inverse PINN  Uniform  N={N}  Re_rec={re_rec:.1f}",
                             f"fig_inverse_uniform_N{N}.png",
                             re_label=f"Re_rec={re_rec:.1f}")

    # ── Step C: N-sweep with WAKE-CLUSTERED sampling ───────────────
    print("\n[C] Inverse PINN — N-sweep with WAKE-CLUSTERED sampling")
    rng_w = np.random.default_rng(13)

    wake_results = []
    converged_N_wake = None

    for N in range(10, 110, 10):
        print(f"\n  N = {N} (wake-clustered)")
        pts_xy  = sample_wake(N, rng=rng_w)
        pts_uvp = generate_observations(fwd_model, pts_xy)

        inv_model_w, re_rec_w, hist_w = train_inverse_pinn(
            pts_xy, pts_uvp,
            re_init=50.0,
            epochs=EPOCHS_INVERSE,
            verbose=True,
            label=f"Wake N={N}"
        )

        err_w = abs(re_rec_w - RE_TRUE) / RE_TRUE * 100
        final_loss_w = hist_w["total"][-1]
        wake_results.append(dict(N=N, strategy="wake",
                                 Re_rec=re_rec_w, error=err_w,
                                 final_loss=final_loss_w))
        print(f"  >> N={N}  Re_rec={re_rec_w:.2f}  Error={err_w:.2f}%  "
              f"Loss={final_loss_w:.4e}")

        plot_loss_history(hist_w, RE_TRUE,
                          f"Wake N={N}",
                          f"fig_loss_wake_N{N:03d}.png")

        if err_w <= CONVERGENCE_TOL * 100 and converged_N_wake is None:
            converged_N_wake = N
            print(f"  *** CONVERGED at N={N} (wake) ***")
            plot_flow_fields(inv_model_w,
                             f"Inverse PINN  Wake  N={N}  Re_rec={re_rec_w:.1f}",
                             f"fig_inverse_wake_N{N}.png",
                             re_label=f"Re_rec={re_rec_w:.1f}")

    # ── Step D: Final comparison plots ────────────────────────────
    print("\n[D] Generating summary plots and tables...")

    n_u   = [r["N"]     for r in sweep_results]
    re_u  = [r["Re_rec"] for r in sweep_results]
    err_u = [r["error"] for r in sweep_results]

    plot_re_convergence(n_u, re_u, err_u,
                        "fig2_re_convergence_uniform.png")
    plot_sampling_comparison(
        converged_N_uniform, sweep_results, wake_results,
        "fig3_sampling_comparison.png"
    )

    # Observation location plot (at max N for illustration)
    obs_u_vis = sample_domain(converged_N_uniform or 40,
                               rng=np.random.default_rng(7))
    obs_w_vis = sample_wake(converged_N_wake or 30,
                             rng=np.random.default_rng(13))
    plot_obs_location(obs_u_vis, obs_w_vis,
                      "fig4_observation_locations.png")

    # ── Step E: Print & save summary tables ───────────────────────
    print("\n" + "="*65)
    print("  TABLE 1 — Uniform Sampling: N vs Recovered Re")
    print("="*65)
    df_u = pd.DataFrame(sweep_results)
    df_u = df_u[["N", "Re_rec", "error", "final_loss", "strategy"]]
    df_u.columns = ["N", "Recovered Re", "% Error", "Final Loss", "Strategy"]
    print(df_u.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    print("\n" + "="*65)
    print("  TABLE 2 — Wake-Clustered Sampling: N vs Recovered Re")
    print("="*65)
    df_w = pd.DataFrame(wake_results)
    df_w = df_w[["N", "Re_rec", "error", "final_loss", "strategy"]]
    df_w.columns = ["N", "Recovered Re", "% Error", "Final Loss", "Strategy"]
    print(df_w.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    # Save to CSV
    df_u.to_csv(os.path.join(OUTPUT_DIR, "table1_uniform_sweep.csv"),
                index=False)
    df_w.to_csv(os.path.join(OUTPUT_DIR, "table2_wake_sweep.csv"),
                index=False)

    # ── Step F: Convergence summary ───────────────────────────────
    print("\n" + "="*65)
    print("  CONVERGENCE SUMMARY")
    print("="*65)
    if converged_N_uniform:
        r = next(x for x in sweep_results if x["N"] == converged_N_uniform)
        print(f"  Uniform sampling  : converged at N={converged_N_uniform}  "
              f"Re_rec={r['Re_rec']:.2f}  "
              f"Error={r['error']:.2f}%")
    else:
        print("  Uniform sampling  : did NOT converge within N=100")

    if converged_N_wake:
        r = next(x for x in wake_results if x["N"] == converged_N_wake)
        print(f"  Wake sampling     : converged at N={converged_N_wake}  "
              f"Re_rec={r['Re_rec']:.2f}  "
              f"Error={r['error']:.2f}%")
    else:
        print("  Wake sampling     : did NOT converge within N=100")

    print("\n  All outputs saved to:", OUTPUT_DIR)
    print("="*65)


if __name__ == "__main__":
    main()